In [1]:
# ============================================================
# PHASE 3 — PANDAS CLEANING & TRANSFORMATION
# ============================================================
#
# BUSINESS OBJECTIVE
# ------------------
# Transform the raw working-capital dataset into a clean,
# reproducible analytical dataset without hiding legitimate
# financial/business anomalies.
#
# IMPORTANT PRINCIPLE
# -------------------
# Cleaning does NOT mean deleting everything unusual.
#
# We will:
#   - fix formatting problems
#   - remove confirmed duplicate records
#   - convert data types
#   - standardize categorical values
#   - preserve legitimate financial anomalies
#   - create quality flags for anomalies
#   - never invent missing financial values
#
# INPUT:
#   Working_capital_dirty .csv
#
# OUTPUT:
#   Working_capital_clean.csv
#   Phase3_Cleaning_Audit.csv
#
# ============================================================


import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. DEFINE INPUT AND OUTPUT PATHS
# ============================================================

input_path = Path(
    r"C:\Users\hp\Downloads\Working_capital_dirty .csv"
)

output_path = Path(
    r"C:\Users\hp\Downloads\Working_capital_clean.csv"
)

audit_path = Path(
    r"C:\Users\hp\Downloads\Phase3_Cleaning_Audit.csv"
)


# ============================================================
# 2. LOAD RAW DATA
# ============================================================

df = pd.read_csv(input_path)

print("=" * 60)
print("PHASE 3 — DATA CLEANING")
print("=" * 60)

print("\nRAW DATASET")
print("-" * 60)
print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns):,}")


# Keep a copy of the raw dataframe in memory.
# The raw CSV itself will never be overwritten.
df_clean = df.copy()


# ============================================================
# 3. RECORD RAW BASELINE
# ============================================================

raw_rows = len(df_clean)
raw_columns = len(df_clean.columns)

raw_exact_duplicates = df_clean.duplicated().sum()

raw_duplicate_invoice_rows = (
    df_clean["invoice_id"].duplicated(keep=False).sum()
)

raw_missing_cells = df_clean.isna().sum().sum()


# ============================================================
# 4. NORMALIZE COLUMN NAMES
# ============================================================
#
# Column names should be predictable before we start building
# transformation logic.
#
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)


# ============================================================
# 5. STANDARDIZE TEXT FIELDS
# ============================================================
#
# First remove accidental leading/trailing whitespace.
#
# We do NOT blindly title-case every column because IDs and
# financial values should not be altered unnecessarily.
#
# ============================================================

text_columns = [
    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "currency",
    "payment_method",
    "status"
]

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype("string")
            .str.strip()
        )


# ============================================================
# 6. CLEAN INVOICE AMOUNT
# ============================================================
#
# The raw invoice_amount column contains:
#
#   - valid numbers stored as text
#   - commas such as "9,501.81"
#   - placeholders such as "unknown"
#   - "not available"
#   - "-"
#   - missing values
#
# We convert valid values to numeric.
# Invalid placeholders become NaN.
#
# We DO NOT replace missing invoice amounts with zero.
#
# ============================================================

invoice_amount_original = df_clean["invoice_amount"].copy()

invoice_amount_text = (
    df_clean["invoice_amount"]
    .astype("string")
    .str.strip()
    .str.replace(",", "", regex=False)
)

# Explicitly identify common placeholder values.
invoice_amount_placeholders = [
    "",
    "-",
    "unknown",
    "not available",
    "n/a",
    "na",
    "null",
    "none"
]

invoice_amount_placeholder_flag = (
    invoice_amount_text
    .str.lower()
    .isin(invoice_amount_placeholders)
)

df_clean["invoice_amount"] = pd.to_numeric(
    invoice_amount_text.mask(
        invoice_amount_placeholder_flag
    ),
    errors="coerce"
)


# ============================================================
# 7. CLEAN PAYMENT AMOUNT
# ============================================================
#
# payment_amount is already numeric in the raw dataset, but
# we explicitly enforce numeric type so the transformation
# remains reproducible if the source file changes later.
#
# Missing payment amounts remain missing.
#
# ============================================================

df_clean["payment_amount"] = pd.to_numeric(
    df_clean["payment_amount"],
    errors="coerce"
)


# ============================================================
# 8. CLEAN DATE COLUMNS
# ============================================================
#
# Invalid dates become NaT.
#
# We do not invent dates when the original value cannot be
# reliably interpreted.
#
# ============================================================

date_columns = [
    "invoice_date",
    "due_date",
    "payment_date"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )


# ============================================================
# 9. STANDARDIZE REGION
# ============================================================
#
# Known formatting variants:
#
#   casablanca
#    casablanca
#   CASABLANCA
#   fes
#   Tanger
#   Marrakech
#   RABAT
#
# These represent the same business categories as the
# standardized values below.
#
# Missing regions remain missing.
#
# ============================================================

df_clean["region"] = (
    df_clean["region"]
    .astype("string")
    .str.strip()
    .str.lower()
)

region_mapping = {
    "casablanca": "Casablanca",
    "fes": "Fes",
    "tanger": "Tangier",
    "marrakech": "Marrakesh",
    "rabat": "Rabat",
    "agadir": "Agadir",
    "oujda": "Oujda"
}

df_clean["region"] = df_clean["region"].replace(
    region_mapping
)


# ============================================================
# 10. STANDARDIZE CURRENCY
# ============================================================
#
# Example:
#
#   "MAD "
#   "MAD"
#
# become:
#
#   "MAD"
#
# Currency is preserved.
#
# We DO NOT convert EUR into MAD because no approved FX
# methodology has been supplied yet.
#
# ============================================================

df_clean["currency"] = (
    df_clean["currency"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ============================================================
# 11. STANDARDIZE PAYMENT METHOD
# ============================================================
#
# Example:
#
#   bank transfer
#   Bank Transfer
#
# become:
#
#   Bank Transfer
#
# ============================================================

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .astype("string")
    .str.strip()
    .str.lower()
)

payment_method_mapping = {
    "bank transfer": "Bank Transfer",
    "cheque": "Cheque",
    "credit card": "Credit Card",
    "cash": "Cash"
}

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .replace(payment_method_mapping)
)


# ============================================================
# 12. STANDARDIZE STATUS
# ============================================================
#
# Status is already clean in the source, but we enforce a
# consistent representation.
#
# ============================================================

df_clean["status"] = (
    df_clean["status"]
    .astype("string")
    .str.strip()
    .str.title()
)


# ============================================================
# 13. REMOVE EXACT DUPLICATE RECORDS
# ============================================================
#
# Phase 2 established:
#
#   - 110 excess exact duplicate rows
#   - 109 duplicated invoice IDs
#   - all duplicated invoice IDs had one distinct business
#     record version
#   - no conflicting duplicate invoice IDs
#
# Therefore keeping the first copy is justified.
#
# We do NOT use invoice_id alone to blindly delete records.
# The Phase 2 consistency analysis already established that
# these duplicates represent repeated copies of the same
# business record.
#
# ============================================================

rows_before_duplicates = len(df_clean)

df_clean = df_clean.drop_duplicates(
    keep="first"
).copy()

rows_after_duplicates = len(df_clean)

duplicates_removed = (
    rows_before_duplicates -
    rows_after_duplicates
)


# ============================================================
# 14. CREATE QUALITY FLAGS
# ============================================================
#
# These flags are extremely important.
#
# We are deliberately NOT deleting unusual financial records.
#
# Instead, we preserve them and make them visible to future
# SQL / Power BI / Analytics Engineering layers.
#
# ============================================================


# ------------------------------------------------------------
# 14.1 Missing invoice amount
# ------------------------------------------------------------

df_clean["flag_missing_invoice_amount"] = (
    df_clean["invoice_amount"].isna()
)


# ------------------------------------------------------------
# 14.2 Invoice amount originally contained an invalid
#     placeholder / non-numeric value
# ------------------------------------------------------------

df_clean["flag_invalid_invoice_amount"] = (
    invoice_amount_placeholder_flag
    | (
        invoice_amount_original.notna()
        & df_clean["invoice_amount"].isna()
    )
)


# ------------------------------------------------------------
# 14.3 Missing customer ID
# ------------------------------------------------------------

df_clean["flag_missing_customer_id"] = (
    df_clean["customer_id"].isna()
)


# ------------------------------------------------------------
# 14.4 Missing region
# ------------------------------------------------------------

df_clean["flag_missing_region"] = (
    df_clean["region"].isna()
)


# ------------------------------------------------------------
# 14.5 Missing payment method
# ------------------------------------------------------------

df_clean["flag_missing_payment_method"] = (
    df_clean["payment_method"].isna()
)


# ------------------------------------------------------------
# 14.6 Invalid / missing invoice date
# ------------------------------------------------------------

df_clean["flag_invalid_invoice_date"] = (
    df_clean["invoice_date"].isna()
)


# ------------------------------------------------------------
# 14.7 Invalid / missing due date
# ------------------------------------------------------------

df_clean["flag_invalid_due_date"] = (
    df_clean["due_date"].isna()
)


# ------------------------------------------------------------
# 14.8 Negative invoice
# ------------------------------------------------------------
#
# We preserve these.
#
# They may represent credit notes, reversals, corrections,
# refunds, or other legitimate accounting transactions.
#
# The dataset does not contain enough information to classify
# them automatically.
#
# ------------------------------------------------------------

df_clean["flag_negative_invoice"] = (
    df_clean["invoice_amount"] < 0
)


# ------------------------------------------------------------
# 14.9 Negative payment
# ------------------------------------------------------------
#
# Same principle: preserve, flag, investigate later.
#
# ------------------------------------------------------------

df_clean["flag_negative_payment"] = (
    df_clean["payment_amount"] < 0
)


# ------------------------------------------------------------
# 14.10 Zero payment
# ------------------------------------------------------------

df_clean["flag_zero_payment"] = (
    df_clean["payment_amount"] == 0
)


# ------------------------------------------------------------
# 14.11 Payment exceeds invoice amount
# ------------------------------------------------------------
#
# IMPORTANT:
# We are NOT changing payment_amount.
#
# Possible explanations could include:
#   - overpayment
#   - credit/adjustment
#   - duplicate payment
#   - data issue
#   - currency issue
#
# This requires business investigation.
#
# ------------------------------------------------------------

df_clean["flag_payment_exceeds_invoice"] = (
    df_clean["payment_amount"].notna()
    & df_clean["invoice_amount"].notna()
    & (
        df_clean["payment_amount"]
        > df_clean["invoice_amount"]
    )
)


# ------------------------------------------------------------
# 14.12 Payment before due date
# ------------------------------------------------------------
#
# This is not automatically an error.
# Early payment can be perfectly legitimate.
#
# ------------------------------------------------------------

df_clean["flag_early_payment"] = (
    df_clean["payment_date"].notna()
    & df_clean["due_date"].notna()
    & (
        df_clean["payment_date"]
        < df_clean["due_date"]
    )
)


# ------------------------------------------------------------
# 14.13 Payment before invoice date
# ------------------------------------------------------------
#
# This is much more suspicious than early payment.
#
# ------------------------------------------------------------

df_clean["flag_payment_before_invoice"] = (
    df_clean["payment_date"].notna()
    & df_clean["invoice_date"].notna()
    & (
        df_clean["payment_date"]
        < df_clean["invoice_date"]
    )
)


# ------------------------------------------------------------
# 14.14 Due date before invoice date
# ------------------------------------------------------------

df_clean["flag_due_before_invoice"] = (
    df_clean["due_date"].notna()
    & df_clean["invoice_date"].notna()
    & (
        df_clean["due_date"]
        < df_clean["invoice_date"]
    )
)


# ------------------------------------------------------------
# 14.15 Paid invoice missing payment amount
# ------------------------------------------------------------

df_clean["flag_paid_missing_payment_amount"] = (
    (df_clean["status"] == "Paid")
    & df_clean["payment_amount"].isna()
)


# ------------------------------------------------------------
# 14.16 Paid invoice missing payment date
# ------------------------------------------------------------

df_clean["flag_paid_missing_payment_date"] = (
    (df_clean["status"] == "Paid")
    & df_clean["payment_date"].isna()
)


# ------------------------------------------------------------
# 14.17 Open invoice has payment evidence
# ------------------------------------------------------------
#
# We do NOT automatically change status.
#
# Instead we create a flag so Phase 4 can investigate the
# business rule.
#
# ------------------------------------------------------------

df_clean["flag_open_with_payment_evidence"] = (
    (df_clean["status"] == "Open")
    & (
        df_clean["payment_amount"].notna()
        | df_clean["payment_date"].notna()
    )
)


# ------------------------------------------------------------
# 14.18 Payment amount exists but payment date is missing
# ------------------------------------------------------------

df_clean["flag_payment_without_date"] = (
    df_clean["payment_amount"].notna()
    & df_clean["payment_date"].isna()
)


# ------------------------------------------------------------
# 14.19 Payment date exists but payment amount is missing
# ------------------------------------------------------------

df_clean["flag_payment_date_without_amount"] = (
    df_clean["payment_date"].notna()
    & df_clean["payment_amount"].isna()
)


# ============================================================
# 15. CREATE A SINGLE OVERALL QUALITY STATUS
# ============================================================
#
# This is NOT a "good/bad transaction" judgment.
#
# It is simply a convenient analytical label for records that
# require attention.
#
# ============================================================

critical_flags = [
    "flag_missing_invoice_amount",
    "flag_paid_missing_payment_amount",
    "flag_paid_missing_payment_date",
    "flag_payment_before_invoice",
    "flag_due_before_invoice"
]

high_flags = [
    "flag_payment_exceeds_invoice",
    "flag_missing_customer_id",
    "flag_invalid_invoice_date",
    "flag_invalid_due_date",
    "flag_open_with_payment_evidence",
    "flag_payment_without_date",
    "flag_payment_date_without_amount"
]

investigate_flags = [
    "flag_negative_invoice",
    "flag_negative_payment",
    "flag_zero_payment",
    "flag_early_payment"
]


df_clean["quality_status"] = "PASS"


critical_mask = df_clean[critical_flags].any(axis=1)
high_mask = df_clean[high_flags].any(axis=1)
investigate_mask = df_clean[investigate_flags].any(axis=1)

df_clean.loc[investigate_mask, "quality_status"] = (
    "INVESTIGATE"
)

df_clean.loc[high_mask, "quality_status"] = (
    "REVIEW"
)

df_clean.loc[critical_mask, "quality_status"] = (
    "CRITICAL"
)


# ============================================================
# 16. CREATE BUSINESS-FRIENDLY DATE DERIVED FIELDS
# ============================================================
#
# These are useful downstream for SQL / Power BI.
#
# We are not calculating DSO/DPO yet.
# That belongs to the analytical layer.
#
# ============================================================

df_clean["invoice_year"] = (
    df_clean["invoice_date"].dt.year
)

df_clean["invoice_month"] = (
    df_clean["invoice_date"].dt.month
)

df_clean["invoice_year_month"] = (
    df_clean["invoice_date"]
    .dt.to_period("M")
    .astype("string")
)


# ============================================================
# 17. CALCULATE DAYS TO PAYMENT
# ============================================================
#
# This is a descriptive field, not yet DSO.
#
# It measures the actual number of calendar days between
# invoice date and payment date.
#
# Missing payment dates remain missing.
#
# ============================================================

df_clean["days_to_payment"] = (
    df_clean["payment_date"]
    - df_clean["invoice_date"]
).dt.days


# ============================================================
# 18. CALCULATE DAYS EARLY / LATE RELATIVE TO DUE DATE
# ============================================================
#
# Positive value:
#   paid after due date
#
# Negative value:
#   paid before due date
#
# Zero:
#   paid exactly on due date
#
# Missing:
#   no payment date
#
# ============================================================

df_clean["days_from_due_date"] = (
    df_clean["payment_date"]
    - df_clean["due_date"]
).dt.days


# ============================================================
# 19. CREATE PAYMENT TIMING CATEGORY
# ============================================================

df_clean["payment_timing"] = np.select(
    [
        df_clean["payment_date"].isna(),

        df_clean["days_from_due_date"] < 0,

        df_clean["days_from_due_date"] == 0,

        df_clean["days_from_due_date"] > 0
    ],
    [
        "Not Paid / No Payment Date",
        "Paid Early",
        "Paid On Time",
        "Paid Late"
    ],
    default="Unknown"
)


# ============================================================
# 20. ORDER COLUMNS
# ============================================================
#
# Keep the core transaction fields first.
# Then analytical fields.
# Then quality flags.
#
# ============================================================

base_columns = [
    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "invoice_date",
    "due_date",
    "payment_date",
    "invoice_amount",
    "payment_amount",
    "currency",
    "payment_method",
    "status"
]

derived_columns = [
    "invoice_year",
    "invoice_month",
    "invoice_year_month",
    "days_to_payment",
    "days_from_due_date",
    "payment_timing"
]

flag_columns = [
    col
    for col in df_clean.columns
    if col.startswith("flag_")
]

final_columns = (
    base_columns
    + derived_columns
    + flag_columns
    + ["quality_status"]
)

df_clean = df_clean[
    [col for col in final_columns if col in df_clean.columns]
]


# ============================================================
# 21. FINAL DATA-TYPE CHECK
# ============================================================

print("\nFINAL DATA TYPES")
print("-" * 60)

print(df_clean.dtypes)


# ============================================================
# 22. BUILD CLEANING AUDIT REPORT
# ============================================================
#
# This report documents what Phase 3 actually changed.
#
# ============================================================

audit_records = [
    {
        "check": "Raw row count",
        "value": raw_rows,
        "action": "Baseline"
    },
    {
        "check": "Clean row count",
        "value": len(df_clean),
        "action": "After transformations"
    },
    {
        "check": "Rows removed as exact duplicates",
        "value": duplicates_removed,
        "action": "Removed"
    },
    {
        "check": "Raw exact duplicate rows",
        "value": raw_exact_duplicates,
        "action": "Diagnosed in Phase 2"
    },
    {
        "check": "Raw duplicated invoice-ID rows",
        "value": raw_duplicate_invoice_rows,
        "action": "Diagnosed in Phase 2"
    },
    {
        "check": "Missing invoice amount after cleaning",
        "value": int(
            df_clean["invoice_amount"].isna().sum()
        ),
        "action": "Preserved as missing"
    },
    {
        "check": "Missing customer ID",
        "value": int(
            df_clean["customer_id"].isna().sum()
        ),
        "action": "Preserved as missing"
    },
    {
        "check": "Missing region",
        "value": int(
            df_clean["region"].isna().sum()
        ),
        "action": "Preserved as missing"
    },
    {
        "check": "Missing payment method",
        "value": int(
            df_clean["payment_method"].isna().sum()
        ),
        "action": "Preserved as missing"
    },
    {
        "check": "Negative invoice records",
        "value": int(
            df_clean["flag_negative_invoice"].sum()
        ),
        "action": "Preserved + flagged"
    },
    {
        "check": "Negative payment records",
        "value": int(
            df_clean["flag_negative_payment"].sum()
        ),
        "action": "Preserved + flagged"
    },
    {
        "check": "Payment exceeds invoice",
        "value": int(
            df_clean["flag_payment_exceeds_invoice"].sum()
        ),
        "action": "Preserved + flagged"
    },
    {
        "check": "Early payment records",
        "value": int(
            df_clean["flag_early_payment"].sum()
        ),
        "action": "Preserved + flagged"
    },
    {
        "check": "Paid missing payment amount",
        "value": int(
            df_clean["flag_paid_missing_payment_amount"].sum()
        ),
        "action": "Preserved + flagged"
    },
    {
        "check": "Open with payment evidence",
        "value": int(
            df_clean["flag_open_with_payment_evidence"].sum()
        ),
        "action": "Preserved + flagged"
    }
]

audit_df = pd.DataFrame(audit_records)


# ============================================================
# 23. SAVE CLEAN DATASET
# ============================================================

df_clean.to_csv(
    output_path,
    index=False
)


# ============================================================
# 24. SAVE AUDIT REPORT
# ============================================================

audit_df.to_csv(
    audit_path,
    index=False
)


# ============================================================
# 25. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("PHASE 3 — FINAL VALIDATION")
print("=" * 60)

print("\nROW COUNTS")
print("-" * 60)
print(f"Raw rows:             {raw_rows:,}")
print(f"Clean rows:           {len(df_clean):,}")
print(f"Rows removed:         {duplicates_removed:,}")

print("\nDUPLICATE CHECK")
print("-" * 60)
print(
    "Exact duplicate rows remaining:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate invoice IDs remaining:",
    df_clean["invoice_id"].duplicated().sum()
)

print("\nNUMERIC CHECK")
print("-" * 60)
print(
    "Invoice amount dtype:",
    df_clean["invoice_amount"].dtype
)

print(
    "Payment amount dtype:",
    df_clean["payment_amount"].dtype
)

print("\nDATE CHECK")
print("-" * 60)

for col in date_columns:
    print(
        f"{col}:",
        df_clean[col].dtype
    )

print("\nQUALITY STATUS")
print("-" * 60)
print(
    df_clean["quality_status"]
    .value_counts(dropna=False)
)

print("\nCURRENCY")
print("-" * 60)
print(
    df_clean["currency"]
    .value_counts(dropna=False)
)

print("\nREGION")
print("-" * 60)
print(
    df_clean["region"]
    .value_counts(dropna=False)
)

print("\nPAYMENT METHOD")
print("-" * 60)
print(
    df_clean["payment_method"]
    .value_counts(dropna=False)
)

print("\nOUTPUT FILES")
print("-" * 60)
print(f"Clean dataset: {output_path}")
print(f"Audit report:  {audit_path}")

print("\n" + "=" * 60)
print("PHASE 3 COMPLETE")
print("=" * 60)

PHASE 3 — DATA CLEANING

RAW DATASET
------------------------------------------------------------
Rows:    12,110
Columns: 14

FINAL DATA TYPES
------------------------------------------------------------
invoice_id                          string[python]
customer_id                         string[python]
customer_name                       string[python]
region                              string[python]
supplier_id                         string[python]
product_id                          string[python]
invoice_date                        datetime64[ns]
due_date                            datetime64[ns]
payment_date                        datetime64[ns]
invoice_amount                             Float64
payment_amount                             float64
currency                            string[python]
payment_method                      string[python]
status                              string[python]
invoice_year                               float64
invoice_month                 

In [7]:
# ============================================================
# PHASE 3 — FINAL CLEANING CORRECTION
# ============================================================
#
# Purpose:
#   Fix the two region representations that were not fully
#   standardized during the first Phase 3 run.
#
#   tangier   -> Tangier
#   marrakesh -> Marrakesh
#
# The existing clean dataset is otherwise preserved.
# ============================================================


import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINE FILE PATHS
# ============================================================

input_path = Path(
    r"C:\Users\hp\Downloads\Working_capital_clean.csv"
)

output_path = Path(
    r"C:\Users\hp\Downloads\Working_capital_clean.csv"
)


# ============================================================
# 2. LOAD CLEAN DATASET
# ============================================================

df_clean = pd.read_csv(input_path)


print("=" * 60)
print("PHASE 3 — FINAL CLEANING CORRECTION")
print("=" * 60)

print("\nBEFORE CORRECTION")
print("-" * 60)

print(
    df_clean["region"]
    .value_counts(dropna=False)
)


# ============================================================
# 3. STANDARDIZE REMAINING REGION VARIANTS
# ============================================================

region_correction = {
    "tangier": "Tangier",
    "marrakesh": "Marrakesh"
}

df_clean["region"] = (
    df_clean["region"]
    .replace(region_correction)
)


# ============================================================
# 4. REBUILD REGION QUALITY FLAG
# ============================================================

df_clean["flag_missing_region"] = (
    df_clean["region"].isna()
)


# ============================================================
# 5. VALIDATE REGION VALUES
# ============================================================

print("\nAFTER CORRECTION")
print("-" * 60)

print(
    df_clean["region"]
    .value_counts(dropna=False)
)


# ============================================================
# 6. CHECK FOR REMAINING REGION VARIANTS
# ============================================================

allowed_regions = {
    "Oujda",
    "Fes",
    "Casablanca",
    "Rabat",
    "Tangier",
    "Agadir",
    "Marrakesh"
}

unexpected_regions = sorted(
    set(
        df_clean["region"]
        .dropna()
        .unique()
    )
    - allowed_regions
)


print("\nREGION VALIDATION")
print("-" * 60)

if len(unexpected_regions) == 0:
    print("No unexpected region values found.")
else:
    print("Unexpected region values:")
    print(unexpected_regions)


# ============================================================
# 7. VERIFY DATASET STRUCTURE
# ============================================================

print("\nDATASET VALIDATION")
print("-" * 60)

print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns):,}")

print(
    "Exact duplicate rows:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate invoice IDs:",
    df_clean["invoice_id"].duplicated().sum()
)


# ============================================================
# 8. SAVE CORRECTED CLEAN DATASET
# ============================================================

df_clean.to_csv(
    output_path,
    index=False
)


# ============================================================
# 9. FINAL MESSAGE
# ============================================================

print("\nOUTPUT")
print("-" * 60)
print(f"Corrected clean dataset saved to:")
print(output_path)

print("\n" + "=" * 60)
print("PHASE 3 CORRECTION COMPLETE")
print("=" * 60)

PHASE 3 — FINAL CLEANING CORRECTION

BEFORE CORRECTION
------------------------------------------------------------
region
Oujda         1758
Fes           1740
Casablanca    1732
Tangier       1671
Rabat         1654
Marrakesh     1645
Agadir        1585
NaN            215
Name: count, dtype: int64

AFTER CORRECTION
------------------------------------------------------------
region
Oujda         1758
Fes           1740
Casablanca    1732
Tangier       1671
Rabat         1654
Marrakesh     1645
Agadir        1585
NaN            215
Name: count, dtype: int64

REGION VALIDATION
------------------------------------------------------------
No unexpected region values found.

DATASET VALIDATION
------------------------------------------------------------
Rows: 12,000
Columns: 40
Exact duplicate rows: 0
Duplicate invoice IDs: 0

OUTPUT
------------------------------------------------------------
Corrected clean dataset saved to:
C:\Users\hp\Downloads\Working_capital_clean.csv

PHASE 3 CORRE

In [6]:
# ============================================================
# PHASE 4 - BUSINESS RULES & VALIDATION
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

RAW_PATH = Path(
    r"C:\Users\hp\Downloads\Working_capital_dirty .csv"
)

OUTPUT_DIR = Path(
    r"C:\Users\hp\Downloads"
)

VALIDATED_PATH = OUTPUT_DIR / "Working_capital_validated.csv"
EXCEPTIONS_PATH = OUTPUT_DIR / "Phase4_Exceptions.csv"
BUSINESS_RULES_PATH = OUTPUT_DIR / "Phase4_Business_Rules_Report.csv"
SUMMARY_PATH = OUTPUT_DIR / "Phase4_Validation_Summary.csv"


# ============================================================
# 2. LOAD RAW DATA
# ============================================================

df = pd.read_csv(RAW_PATH)

print("=" * 70)
print("PHASE 4 CORRECTION — START")
print("=" * 70)

print(f"Input file : {RAW_PATH}")
print(f"Input shape: {df.shape}")


# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

expected_columns = [
    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "invoice_date",
    "due_date",
    "payment_date",
    "invoice_amount",
    "payment_amount",
    "currency",
    "payment_method",
    "status",
]

missing_columns = [
    col
    for col in expected_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}"
    )


# ============================================================
# 4. PRESERVE RAW VALUES BEFORE ANY ROW REMOVAL
# ============================================================
#
# IMPORTANT:
# These temporary columns are created INSIDE df.
# Therefore, when duplicate rows are removed later,
# the raw values stay correctly aligned with their records.
# ============================================================

df["_raw_invoice_amount"] = (
    df["invoice_amount"]
    .astype("string")
    .str.strip()
)

df["_raw_invoice_date"] = (
    df["invoice_date"]
    .astype("string")
    .str.strip()
)

df["_raw_due_date"] = (
    df["due_date"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 5. IDENTIFY TRUE MISSING INVOICE AMOUNTS
# ============================================================

df["_invoice_amount_is_missing"] = (
    df["_raw_invoice_amount"].isna()
    | df["_raw_invoice_amount"].eq("")
)


# ============================================================
# 6. PARSE INVOICE AMOUNT
# ============================================================

invoice_amount_for_parsing = (
    df["_raw_invoice_amount"]
    .str.replace(",", "", regex=False)
)

invoice_amount_for_parsing = invoice_amount_for_parsing.mask(
    invoice_amount_for_parsing.str.lower().isin(
        [
            "not available",
            "unknown",
            "-",
            "n/a",
            "na",
        ]
    )
)

df["_invoice_amount_numeric"] = pd.to_numeric(
    invoice_amount_for_parsing,
    errors="coerce"
)


# ============================================================
# 7. DISTINGUISH MISSING VS INVALID
# ============================================================

df["flag_missing_invoice_amount"] = (
    df["_invoice_amount_is_missing"]
).astype("boolean")


df["flag_invalid_invoice_amount"] = (
    ~df["_invoice_amount_is_missing"]
    & df["_invoice_amount_numeric"].isna()
).astype("boolean")


df["flag_unusable_invoice_amount"] = (
    df["flag_missing_invoice_amount"]
    | df["flag_invalid_invoice_amount"]
).astype("boolean")


# ============================================================
# 8. ASSIGN CLEAN INVOICE AMOUNT
# ============================================================

df["invoice_amount"] = (
    df["_invoice_amount_numeric"]
)


# ============================================================
# 9. PARSE PAYMENT AMOUNT
# ============================================================

df["payment_amount"] = pd.to_numeric(
    df["payment_amount"],
    errors="coerce"
)


# ============================================================
# 10. PARSE DATES
# ============================================================

df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

df["due_date"] = pd.to_datetime(
    df["due_date"],
    errors="coerce"
)

df["payment_date"] = pd.to_datetime(
    df["payment_date"],
    errors="coerce"
)


# ============================================================
# 11. DATE INVALIDITY FLAGS
# ============================================================

invoice_date_was_present = (
    df["_raw_invoice_date"].notna()
    & df["_raw_invoice_date"].ne("")
)

due_date_was_present = (
    df["_raw_due_date"].notna()
    & df["_raw_due_date"].ne("")
)


df["flag_invalid_invoice_date"] = (
    invoice_date_was_present
    & df["invoice_date"].isna()
).astype("boolean")


df["flag_invalid_due_date"] = (
    due_date_was_present
    & df["due_date"].isna()
).astype("boolean")


# ============================================================
# 12. STANDARDIZE TEXT FIELDS
# ============================================================

df["region"] = (
    df["region"]
    .astype("string")
    .str.strip()
    .str.title()
)

df["region"] = df["region"].replace(
    {
        "Tanger": "Tangier",
        "Marrakech": "Marrakesh",
    }
)


df["currency"] = (
    df["currency"]
    .astype("string")
    .str.strip()
    .str.upper()
)


df["payment_method"] = (
    df["payment_method"]
    .astype("string")
    .str.strip()
    .str.title()
)


df["status"] = (
    df["status"]
    .astype("string")
    .str.strip()
    .str.title()
)


df["customer_name"] = (
    df["customer_name"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 13. REMOVE EXACT DUPLICATES
# ============================================================

rows_before_duplicates = len(df)

df = (
    df
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

rows_after_duplicates = len(df)

exact_duplicates_removed = (
    rows_before_duplicates
    - rows_after_duplicates
)

print()
print(
    f"Exact duplicate rows removed: "
    f"{exact_duplicates_removed}"
)


# ============================================================
# 14. MISSING REFERENCE DATA FLAGS
# ============================================================

df["flag_missing_customer_id"] = (
    df["customer_id"].isna()
).astype("boolean")


df["flag_missing_region"] = (
    df["region"].isna()
).astype("boolean")


df["flag_missing_payment_method"] = (
    df["payment_method"].isna()
).astype("boolean")


# ============================================================
# 15. FINANCIAL FLAGS
# ============================================================

df["flag_negative_invoice"] = (
    df["invoice_amount"] < 0
).astype("boolean")

# Unknown invoice amount means we cannot determine
# whether the invoice is negative.

df.loc[
    df["invoice_amount"].isna(),
    "flag_negative_invoice"
] = pd.NA


df["flag_negative_payment"] = (
    df["payment_amount"] < 0
).astype("boolean")


df["flag_zero_payment"] = (
    df["payment_amount"] == 0
).astype("boolean")


# IMPORTANT:
# fillna(False) makes this a normal boolean condition
# that NumPy can safely process.

df["flag_payment_exceeds_invoice"] = (
    (
        df["payment_amount"].notna()
        & df["invoice_amount"].notna()
        & (
            df["payment_amount"]
            > df["invoice_amount"]
        )
    )
    .fillna(False)
    .astype("boolean")
)


# ============================================================
# 16. DATE RELATIONSHIP FLAGS
# ============================================================

df["flag_early_payment"] = (
    (
        df["payment_date"].notna()
        & df["due_date"].notna()
        & (
            df["payment_date"]
            < df["due_date"]
        )
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_payment_before_invoice"] = (
    (
        df["payment_date"].notna()
        & df["invoice_date"].notna()
        & (
            df["payment_date"]
            < df["invoice_date"]
        )
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_due_before_invoice"] = (
    (
        df["due_date"].notna()
        & df["invoice_date"].notna()
        & (
            df["due_date"]
            < df["invoice_date"]
        )
    )
    .fillna(False)
    .astype("boolean")
)


# ============================================================
# 17. STATUS / PAYMENT-EVIDENCE FLAGS
# ============================================================

df["flag_paid_missing_payment_amount"] = (
    (
        (df["status"] == "Paid")
        & df["payment_amount"].isna()
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_paid_missing_payment_date"] = (
    (
        (df["status"] == "Paid")
        & df["payment_date"].isna()
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_open_with_payment_evidence"] = (
    (
        (df["status"] == "Open")
        & (
            df["payment_amount"].notna()
            | df["payment_date"].notna()
        )
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_payment_without_date"] = (
    (
        df["payment_amount"].notna()
        & df["payment_date"].isna()
    )
    .fillna(False)
    .astype("boolean")
)


df["flag_payment_date_without_amount"] = (
    (
        df["payment_date"].notna()
        & df["payment_amount"].isna()
    )
    .fillna(False)
    .astype("boolean")
)


# ============================================================
# 18. ORIGINAL STATUS
# ============================================================

df["original_status"] = df["status"]


# ============================================================
# 19. DERIVED DATE FIELDS
# ============================================================

df["invoice_year"] = (
    df["invoice_date"].dt.year
)

df["invoice_month"] = (
    df["invoice_date"].dt.month
)

df["invoice_year_month"] = (
    df["invoice_date"]
    .dt.to_period("M")
    .astype("string")
)


# ============================================================
# 20. PAYMENT TIMING METRICS
# ============================================================

df["days_to_payment"] = (
    df["payment_date"]
    - df["invoice_date"]
).dt.days


df["days_from_due_date"] = (
    df["payment_date"]
    - df["due_date"]
).dt.days


df["payment_timing"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)


early_mask = (
    (
        df["payment_date"].notna()
        & df["due_date"].notna()
        & (
            df["payment_date"]
            < df["due_date"]
        )
    )
    .fillna(False)
)


on_due_mask = (
    (
        df["payment_date"].notna()
        & df["due_date"].notna()
        & (
            df["payment_date"]
            == df["due_date"]
        )
    )
    .fillna(False)
)


late_mask = (
    (
        df["payment_date"].notna()
        & df["due_date"].notna()
        & (
            df["payment_date"]
            > df["due_date"]
        )
    )
    .fillna(False)
)


df.loc[
    early_mask,
    "payment_timing"
] = "Early"


df.loc[
    on_due_mask,
    "payment_timing"
] = "On Due Date"


df.loc[
    late_mask,
    "payment_timing"
] = "Late"


# ============================================================
# 21. PAYMENT RECONCILIATION
# ============================================================
#
# FIX:
# Every condition is explicitly converted to ordinary
# True/False values before passing to np.select().
# ============================================================

invoice = df["invoice_amount"]
payment = df["payment_amount"]


reconciliation_conditions = [
    invoice.isna().to_numpy(dtype=bool),

    payment.isna().to_numpy(dtype=bool),

    (
        payment < 0
    ).fillna(False).to_numpy(dtype=bool),

    (
        payment == 0
    ).fillna(False).to_numpy(dtype=bool),

    (
        payment > invoice
    ).fillna(False).to_numpy(dtype=bool),

    (
        payment == invoice
    ).fillna(False).to_numpy(dtype=bool),

    (
        payment < invoice
    ).fillna(False).to_numpy(dtype=bool),
]


reconciliation_choices = [
    "Missing Invoice Amount",
    "Missing Payment Amount",
    "Negative Payment",
    "Zero Payment",
    "Payment Exceeds Invoice",
    "Fully Matched",
    "Partial Payment",
]


df["payment_reconciliation"] = np.select(
    reconciliation_conditions,
    reconciliation_choices,
    default="Other"
)


# ============================================================
# 22. QUALITY ISSUE COUNT
# ============================================================

flag_columns = [
    "flag_missing_invoice_amount",
    "flag_invalid_invoice_amount",
    "flag_unusable_invoice_amount",
    "flag_missing_customer_id",
    "flag_missing_region",
    "flag_missing_payment_method",
    "flag_invalid_invoice_date",
    "flag_invalid_due_date",
    "flag_negative_invoice",
    "flag_negative_payment",
    "flag_zero_payment",
    "flag_payment_exceeds_invoice",
    "flag_early_payment",
    "flag_payment_before_invoice",
    "flag_due_before_invoice",
    "flag_paid_missing_payment_amount",
    "flag_paid_missing_payment_date",
    "flag_open_with_payment_evidence",
    "flag_payment_without_date",
    "flag_payment_date_without_amount",
]


for col in flag_columns:
    df[col] = df[col].astype("boolean")


df["quality_issue_count"] = (
    df[flag_columns]
    .fillna(False)
    .sum(axis=1)
)


# ============================================================
# 23. BUSINESS RULE STATUS
# ============================================================

incomplete_mask = (
    df["flag_unusable_invoice_amount"]
    .fillna(False)
)


review_mask = (
    df["flag_payment_exceeds_invoice"].fillna(False)
    | df["flag_negative_invoice"].fillna(False)
    | df["flag_negative_payment"].fillna(False)
    | df["flag_zero_payment"].fillna(False)
    | df["flag_paid_missing_payment_amount"].fillna(False)
    | df["flag_paid_missing_payment_date"].fillna(False)
    | df["flag_open_with_payment_evidence"].fillna(False)
    | df["flag_payment_without_date"].fillna(False)
    | df["flag_payment_date_without_amount"].fillna(False)
)


expected_mask = (
    (df["status"] == "Open")
    & df["payment_amount"].isna()
    & df["payment_date"].isna()
    & ~incomplete_mask
    & ~review_mask
)


# Convert every mask to normal NumPy boolean arrays.

incomplete_np = (
    incomplete_mask
    .fillna(False)
    .to_numpy(dtype=bool)
)

review_np = (
    review_mask
    .fillna(False)
    .to_numpy(dtype=bool)
)

expected_np = (
    expected_mask
    .fillna(False)
    .to_numpy(dtype=bool)
)


df["business_rule_status"] = np.select(
    [
        incomplete_np,
        review_np,
        expected_np,
    ],
    [
        "INCOMPLETE",
        "REVIEW",
        "EXPECTED",
    ],
    default="VALID"
)


# ============================================================
# 24. FINANCIAL ANALYTICAL ELIGIBILITY
# ============================================================

full_mask = (
    df["invoice_amount"].notna()
    & df["payment_amount"].notna()
)

partial_mask = (
    df["invoice_amount"].notna()
    & df["payment_amount"].isna()
)


df["financial_analytical_eligibility"] = np.select(
    [
        df["invoice_amount"].isna().to_numpy(dtype=bool),

        partial_mask
        .fillna(False)
        .to_numpy(dtype=bool),

        full_mask
        .fillna(False)
        .to_numpy(dtype=bool),
    ],
    [
        "NOT_USABLE_FOR_AMOUNT_ANALYSIS",
        "PARTIAL",
        "FULL",
    ],
    default="NOT_USABLE_FOR_AMOUNT_ANALYSIS"
)


# ============================================================
# 25. BUSINESS RULE REASONS
# ============================================================

reason_conditions = [
    (
        df["flag_missing_invoice_amount"],
        "MISSING_INVOICE_AMOUNT"
    ),

    (
        df["flag_invalid_invoice_amount"],
        "INVALID_INVOICE_AMOUNT"
    ),

    (
        df["flag_missing_customer_id"],
        "MISSING_CUSTOMER_ID"
    ),

    (
        df["flag_missing_region"],
        "MISSING_REGION"
    ),

    (
        df["flag_missing_payment_method"],
        "MISSING_PAYMENT_METHOD"
    ),

    (
        df["flag_negative_invoice"],
        "NEGATIVE_INVOICE"
    ),

    (
        df["flag_negative_payment"],
        "NEGATIVE_PAYMENT"
    ),

    (
        df["flag_zero_payment"],
        "ZERO_PAYMENT"
    ),

    (
        df["flag_payment_exceeds_invoice"],
        "PAYMENT_EXCEEDS_INVOICE"
    ),

    (
        df["flag_early_payment"],
        "EARLY_PAYMENT"
    ),

    (
        df["flag_payment_before_invoice"],
        "PAYMENT_BEFORE_INVOICE"
    ),

    (
        df["flag_due_before_invoice"],
        "DUE_DATE_BEFORE_INVOICE"
    ),

    (
        df["flag_paid_missing_payment_amount"],
        "PAID_MISSING_PAYMENT_AMOUNT"
    ),

    (
        df["flag_paid_missing_payment_date"],
        "PAID_MISSING_PAYMENT_DATE"
    ),

    (
        df["flag_open_with_payment_evidence"],
        "OPEN_WITH_PAYMENT_EVIDENCE"
    ),

    (
        df["flag_payment_without_date"],
        "PAYMENT_WITHOUT_DATE"
    ),

    (
        df["flag_payment_date_without_amount"],
        "PAYMENT_DATE_WITHOUT_AMOUNT"
    ),

    (
        df["flag_invalid_invoice_date"],
        "INVALID_INVOICE_DATE"
    ),

    (
        df["flag_invalid_due_date"],
        "INVALID_DUE_DATE"
    ),
]


reason_parts = []

for mask, reason in reason_conditions:

    safe_mask = (
        mask
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    reason_parts.append(
        pd.Series(
            np.where(
                safe_mask,
                reason,
                ""
            ),
            index=df.index
        )
    )


reason_df = pd.concat(
    reason_parts,
    axis=1
)


df["business_rule_reason"] = (
    reason_df
    .fillna("")
    .agg(
        lambda row: ";".join(
            value
            for value in row
            if value != ""
        ),
        axis=1
    )
)


df["business_rule_reason"] = (
    df["business_rule_reason"]
    .replace(
        "",
        "NO_EXCEPTION"
    )
)


# ============================================================
# 26. DOWNSTREAM ANALYTICAL FLAGS
# ============================================================

df["is_invoice_amount_usable"] = (
    df["invoice_amount"].notna()
).astype("boolean")


df["is_payment_amount_usable"] = (
    df["payment_amount"].notna()
).astype("boolean")


df["has_payment_evidence"] = (
    df["payment_amount"].notna()
    | df["payment_date"].notna()
).astype("boolean")


df["is_outstanding_receivable_candidate"] = (
    (df["status"] == "Open")
    & df["invoice_amount"].notna()
).astype("boolean")


df["is_payment_performance_candidate"] = (
    (df["status"] == "Paid")
    & df["invoice_amount"].notna()
    & df["payment_date"].notna()
).astype("boolean")


# ============================================================
# 27. QUALITY STATUS
# ============================================================

quality_critical = (
    df["flag_unusable_invoice_amount"]
    .fillna(False)
)

quality_investigate = review_mask.fillna(False)

quality_review = (
    df["flag_missing_customer_id"].fillna(False)
    | df["flag_missing_region"].fillna(False)
    | df["flag_missing_payment_method"].fillna(False)
    | df["flag_invalid_invoice_date"].fillna(False)
    | df["flag_invalid_due_date"].fillna(False)
)


df["quality_status"] = np.select(
    [
        quality_critical.to_numpy(dtype=bool),
        quality_investigate.to_numpy(dtype=bool),
        quality_review.to_numpy(dtype=bool),
    ],
    [
        "CRITICAL",
        "INVESTIGATE",
        "REVIEW",
    ],
    default="PASS"
)


# ============================================================
# 28. REORDER CORE COLUMNS
# ============================================================

core_columns = [
    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "invoice_date",
    "due_date",
    "payment_date",
    "invoice_amount",
    "payment_amount",
    "currency",
    "payment_method",
    "status",
]


temporary_columns = [
    "_raw_invoice_amount",
    "_raw_invoice_date",
    "_raw_due_date",
    "_invoice_amount_is_missing",
    "_invoice_amount_numeric",
]


remaining_columns = [
    col
    for col in df.columns
    if col not in core_columns
    and col not in temporary_columns
]


df = df[
    core_columns
    + remaining_columns
]


# ============================================================
# 29. VALIDATION CHECKS
# ============================================================

validation_results = []


def record_check(
    check_name,
    condition,
    details
):
    validation_results.append(
        {
            "check": check_name,
            "result": "PASS" if condition else "FAIL",
            "details": details,
        }
    )


# ------------------------------------------------------------
# Row count
# ------------------------------------------------------------

record_check(
    "Row count after duplicate removal",
    len(df) == 12000,
    f"Rows = {len(df)}"
)


# ------------------------------------------------------------
# Invoice IDs
# ------------------------------------------------------------

record_check(
    "Invoice IDs are unique",
    df["invoice_id"].is_unique,
    f"Unique invoice IDs = {df['invoice_id'].nunique()}"
)


record_check(
    "Invoice ID has no missing values",
    df["invoice_id"].notna().all(),
    f"Missing invoice IDs = {df['invoice_id'].isna().sum()}"
)


# ------------------------------------------------------------
# Exact duplicates
# ------------------------------------------------------------

record_check(
    "No exact duplicate rows remain",
    not df.duplicated().any(),
    f"Duplicate rows = {df.duplicated().sum()}"
)


# ------------------------------------------------------------
# Date relationships
# ------------------------------------------------------------

payment_before_invoice = (
    df["payment_date"].notna()
    & df["invoice_date"].notna()
    & (
        df["payment_date"]
        < df["invoice_date"]
    )
).sum()


record_check(
    "No payment before invoice",
    payment_before_invoice == 0,
    f"Violations = {payment_before_invoice}"
)


due_before_invoice = (
    df["due_date"].notna()
    & df["invoice_date"].notna()
    & (
        df["due_date"]
        < df["invoice_date"]
    )
).sum()


record_check(
    "No due date before invoice",
    due_before_invoice == 0,
    f"Violations = {due_before_invoice}"
)


# ------------------------------------------------------------
# Categories
# ------------------------------------------------------------

allowed_status = {
    "Open",
    "Paid"
}

unexpected_status = sorted(
    set(df["status"].dropna().unique())
    - allowed_status
)


record_check(
    "Status categories are valid",
    len(unexpected_status) == 0,
    f"Unexpected = {unexpected_status}"
)


allowed_currency = {
    "MAD",
    "EUR"
}

unexpected_currency = sorted(
    set(df["currency"].dropna().unique())
    - allowed_currency
)


record_check(
    "Currency categories are valid",
    len(unexpected_currency) == 0,
    f"Unexpected = {unexpected_currency}"
)


allowed_regions = {
    "Casablanca",
    "Rabat",
    "Tangier",
    "Marrakesh",
    "Fes",
    "Agadir",
    "Oujda",
}

unexpected_regions = sorted(
    set(df["region"].dropna().unique())
    - allowed_regions
)


record_check(
    "Region categories are standardized",
    len(unexpected_regions) == 0,
    f"Unexpected = {unexpected_regions}"
)


# ============================================================
# 30. CRITICAL INVOICE-AMOUNT VALIDATION
# ============================================================

missing_invoice_count = int(
    df["flag_missing_invoice_amount"].sum()
)

invalid_invoice_count = int(
    df["flag_invalid_invoice_amount"].sum()
)

unusable_invoice_count = int(
    df["flag_unusable_invoice_amount"].sum()
)


record_check(
    "Missing invoice amount count is 30",
    missing_invoice_count == 30,
    f"Missing = {missing_invoice_count}"
)


record_check(
    "Invalid invoice amount count is 80",
    invalid_invoice_count == 80,
    f"Invalid = {invalid_invoice_count}"
)


record_check(
    "Unusable invoice amount count is 110",
    unusable_invoice_count == 110,
    f"Unusable = {unusable_invoice_count}"
)


# ------------------------------------------------------------
# Negative invoice
# ------------------------------------------------------------

negative_invoice_count = int(
    df["flag_negative_invoice"]
    .fillna(False)
    .sum()
)


negative_invoice_unknown = int(
    df["flag_negative_invoice"]
    .isna()
    .sum()
)


record_check(
    "Negative invoice count is 79",
    negative_invoice_count == 79,
    f"Negative = {negative_invoice_count}"
)


record_check(
    "Negative invoice is unknown for unusable amounts",
    (
        negative_invoice_unknown == 110
        and
        df.loc[
            df["flag_negative_invoice"].isna(),
            "invoice_amount"
        ].isna().all()
    ),
    f"Unknown negative status = {negative_invoice_unknown}"
)


# ------------------------------------------------------------
# Business rule reason
# ------------------------------------------------------------

blank_reason_count = int(
    (
        df["business_rule_reason"].isna()
        | df["business_rule_reason"].eq("")
    ).sum()
)


record_check(
    "Business rule reason has no blanks",
    blank_reason_count == 0,
    f"Blank reasons = {blank_reason_count}"
)


# ============================================================
# 31. VALIDATION RESULTS
# ============================================================

validation_df = pd.DataFrame(
    validation_results
)


print()
print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(
    validation_df.to_string(index=False)
)


if (
    validation_df["result"] == "FAIL"
).any():

    raise ValueError(
        "One or more Phase 4 validation checks failed."
    )


# ============================================================
# 32. EXCEPTION DATASET
# ============================================================
#
# Exceptions are NOT deleted from the main dataset.
# This is only a review sidecar.
# ============================================================

exception_mask = (
    df["business_rule_status"].isin(
        [
            "INCOMPLETE",
            "REVIEW",
        ]
    )
)


exceptions_df = df.loc[
    exception_mask
].copy()


# ============================================================
# 33. BUSINESS RULE REPORT
# ============================================================

business_rule_report = pd.DataFrame(
    [
        {
            "rule": "Missing invoice amount",
            "count": missing_invoice_count,
            "severity": "CRITICAL",
            "interpretation":
                "Invoice amount was genuinely absent.",
        },

        {
            "rule": "Invalid invoice amount",
            "count": invalid_invoice_count,
            "severity": "CRITICAL",
            "interpretation":
                "Invoice amount existed but could not be parsed.",
        },

        {
            "rule": "Unusable invoice amount",
            "count": unusable_invoice_count,
            "severity": "CRITICAL",
            "interpretation":
                "Missing plus invalid invoice amounts.",
        },

        {
            "rule": "Negative invoice",
            "count": negative_invoice_count,
            "severity": "INVESTIGATE",
            "interpretation":
                "Negative amount may represent a credit note, reversal, or error.",
        },

        {
            "rule": "Negative payment",
            "count": int(
                df["flag_negative_payment"].sum()
            ),
            "severity": "INVESTIGATE",
            "interpretation":
                "Payment amount is negative.",
        },

        {
            "rule": "Zero payment",
            "count": int(
                df["flag_zero_payment"].sum()
            ),
            "severity": "INVESTIGATE",
            "interpretation":
                "Payment amount is zero.",
        },

        {
            "rule": "Payment exceeds invoice",
            "count": int(
                df["flag_payment_exceeds_invoice"].sum()
            ),
            "severity": "HIGH",
            "interpretation":
                "Payment exceeds invoice amount.",
        },

        {
            "rule": "Early payment",
            "count": int(
                df["flag_early_payment"].sum()
            ),
            "severity": "INVESTIGATE",
            "interpretation":
                "Payment occurred before due date.",
        },

        {
            "rule": "Open with payment evidence",
            "count": int(
                df["flag_open_with_payment_evidence"].sum()
            ),
            "severity": "HIGH",
            "interpretation":
                "Open invoice contains payment evidence.",
        },

        {
            "rule": "Paid missing payment amount",
            "count": int(
                df["flag_paid_missing_payment_amount"].sum()
            ),
            "severity": "HIGH",
            "interpretation":
                "Paid invoice has no payment amount.",
        },

        {
            "rule": "Missing customer ID",
            "count": int(
                df["flag_missing_customer_id"].sum()
            ),
            "severity": "HIGH",
            "interpretation":
                "Customer ID could not be recovered safely.",
        },

        {
            "rule": "Missing region",
            "count": int(
                df["flag_missing_region"].sum()
            ),
            "severity": "MEDIUM",
            "interpretation":
                "Region could not be recovered safely.",
        },

        {
            "rule": "Missing payment method",
            "count": int(
                df["flag_missing_payment_method"].sum()
            ),
            "severity": "MEDIUM",
            "interpretation":
                "Payment method could not be recovered safely.",
        },
    ]
)


# ============================================================
# 34. VALIDATION SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    [
        {
            "metric": "Input rows",
            "value": rows_before_duplicates,
        },

        {
            "metric": "Exact duplicates removed",
            "value": exact_duplicates_removed,
        },

        {
            "metric": "Validated rows",
            "value": len(df),
        },

        {
            "metric": "Unique invoice IDs",
            "value": df["invoice_id"].nunique(),
        },

        {
            "metric": "Missing invoice amounts",
            "value": missing_invoice_count,
        },

        {
            "metric": "Invalid invoice amounts",
            "value": invalid_invoice_count,
        },

        {
            "metric": "Unusable invoice amounts",
            "value": unusable_invoice_count,
        },

        {
            "metric": "Negative invoices",
            "value": negative_invoice_count,
        },

        {
            "metric": "Missing customer IDs",
            "value": int(
                df["flag_missing_customer_id"].sum()
            ),
        },

        {
            "metric": "Missing regions",
            "value": int(
                df["flag_missing_region"].sum()
            ),
        },

        {
            "metric": "Missing payment methods",
            "value": int(
                df["flag_missing_payment_method"].sum()
            ),
        },

        {
            "metric": "Payment exceeds invoice",
            "value": int(
                df["flag_payment_exceeds_invoice"].sum()
            ),
        },

        {
            "metric": "Early payments",
            "value": int(
                df["flag_early_payment"].sum()
            ),
        },

        {
            "metric": "Business status VALID",
            "value": int(
                (df["business_rule_status"] == "VALID").sum()
            ),
        },

        {
            "metric": "Business status REVIEW",
            "value": int(
                (df["business_rule_status"] == "REVIEW").sum()
            ),
        },

        {
            "metric": "Business status EXPECTED",
            "value": int(
                (df["business_rule_status"] == "EXPECTED").sum()
            ),
        },

        {
            "metric": "Business status INCOMPLETE",
            "value": int(
                (df["business_rule_status"] == "INCOMPLETE").sum()
            ),
        },
    ]
)


# ============================================================
# 35. SAVE OUTPUTS
# ============================================================

df.to_csv(
    VALIDATED_PATH,
    index=False
)

exceptions_df.to_csv(
    EXCEPTIONS_PATH,
    index=False
)

business_rule_report.to_csv(
    BUSINESS_RULES_PATH,
    index=False
)

validation_df.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("PHASE 4 CORRECTION — COMPLETE")
print("=" * 70)

print()
print("Dataset:")
print(
    f"  Rows                    : {len(df):,}"
)
print(
    f"  Columns                 : {len(df.columns):,}"
)
print(
    f"  Exact duplicates removed: "
    f"{exact_duplicates_removed:,}"
)
print(
    f"  Unique invoice IDs      : "
    f"{df['invoice_id'].nunique():,}"
)

print()
print("Invoice amount classification:")
print(
    f"  Missing                 : "
    f"{missing_invoice_count:,}"
)
print(
    f"  Invalid                 : "
    f"{invalid_invoice_count:,}"
)
print(
    f"  Unusable                : "
    f"{unusable_invoice_count:,}"
)
print(
    f"  Valid numeric           : "
    f"{df['invoice_amount'].notna().sum():,}"
)

print()
print("Business rule status:")
print(
    df["business_rule_status"]
    .value_counts()
    .to_string()
)

print()
print("Financial analytical eligibility:")
print(
    df["financial_analytical_eligibility"]
    .value_counts()
    .to_string()
)

print()
print("Blank business-rule reasons:")
print(
    (
        df["business_rule_reason"].isna()
        | df["business_rule_reason"].eq("")
    ).sum()
)

print()
print("Files saved:")
print(f"  {VALIDATED_PATH}")
print(f"  {EXCEPTIONS_PATH}")
print(f"  {BUSINESS_RULES_PATH}")
print(f"  {SUMMARY_PATH}")

print()
print("PHASE 4 CORRECTION COMPLETE.")
print("Do NOT move to Phase 5 until the results above are reviewed.")

PHASE 4 CORRECTION — START
Input file : C:\Users\hp\Downloads\Working_capital_dirty .csv
Input shape: (12110, 14)

Exact duplicate rows removed: 110

VALIDATION RESULTS
                                           check result                       details
               Row count after duplicate removal   PASS                  Rows = 12000
                          Invoice IDs are unique   PASS    Unique invoice IDs = 12000
                Invoice ID has no missing values   PASS       Missing invoice IDs = 0
                  No exact duplicate rows remain   PASS            Duplicate rows = 0
                       No payment before invoice   PASS                Violations = 0
                      No due date before invoice   PASS                Violations = 0
                     Status categories are valid   PASS               Unexpected = []
                   Currency categories are valid   PASS               Unexpected = []
              Region categories are standardized   PASS  